In [11]:
# imports
import pandas as pd

In [12]:
# Ruta relativa del xslx que contiene los datos
original_data_path = 'Data\\Original\\PERSONAS_DEMOGRAFICO_Cuadros_CNPV_2018.xlsx'

# Nombre del csv objetivo
csv_total_filename = 'caldas_viterbo_total_3PM.csv'
csv_filename = 'caldas_viterbo_3PM.csv'

# Ruta relativa del csv objetivo
csv_total_path = 'Data\\Datasets\\' + csv_total_filename
csv_path = 'Data\\Datasets\\' + csv_filename

In [13]:
# Lee archivo xlsx
df_3pm = pd.read_excel(original_data_path, sheet_name='3PM', skiprows=9)
df_3pm.head()

,Muniicpios y edades simples,Unnamed: 1,Unnamed: 2,Unnamed: 3,Total,Sexo,Unnamed: 6,Cabecera,Unnamed: 8,Unnamed: 9,Centro Poblado,Unnamed: 11,Unnamed: 12,Rural disperso,Unnamed: 14,Unnamed: 15
0,NaN,NaN,NaN,NaN,NaN,Hombre,Mujer,Total,Hombre,Mujer,Total,Hombre,Mujer,Total,Hombre,Mujer
1,Total,NaN,0 a 4,0,560377.0,286536,273841,410654,210110,200544,47548,24335,23213,102175,52091,50084
2,NaN,NaN,NaN,1,602549.0,308380,294169,438701,224544,214157,50452,25911,24541,113396,57925,55471
3,NaN,NaN,NaN,2,621308.0,318301,303007,455649,233499,222150,51122,26232,24890,114537,58570,55967
4,NaN,NaN,NaN,3,626208.0,320496,305712,458882,234641,224241,51264,26222,25042,116062,59633,56429


In [14]:
first_row = list(df_3pm.iloc[0])[:]
for i, value in enumerate(first_row):
    print(i, value)

0 nan
1 nan
2 nan
3 nan
4 nan
5 Hombre
6 Mujer
7 Total
8 Hombre
9 Mujer
10 Total
11 Hombre
12 Mujer
13 Total
14 Hombre
15 Mujer


Se observan columnas agrupadas por área, las subcolumnas contienen los headers para el dataset.

In [16]:
actual_headers = list(df_3pm.iloc[0][4:7])
actual_headers[0] = 'Total'
actual_headers

['Total', 'Hombre', 'Mujer']

Se filtrarán los datos que no corresponden al municipio de Viterbo Caldas, para ello se propagará el municipio, el cual se encuentra en la columna "Unnamed: 1" y se seleccionarán solo los registros que contengan el identificador de Viterbo dentro del dataframe, para este caso es "17877_Viterbo".

In [17]:
df_3pm['Unnamed: 1'] = df_3pm['Unnamed: 1'].ffill()
df_3pm = df_3pm[df_3pm['Unnamed: 1'] == '17877_Viterbo']
df_3pm

,Muniicpios y edades simples,Unnamed: 1,Unnamed: 2,Unnamed: 3,Total,Sexo,Unnamed: 6,Cabecera,Unnamed: 8,Unnamed: 9,Centro Poblado,Unnamed: 11,Unnamed: 12,Rural disperso,Unnamed: 14,Unnamed: 15
42474,NaN,17877_Viterbo,0 a 4,0,119.0,63,56,95,53,42,0,0,0,24,10,14
42475,NaN,17877_Viterbo,NaN,1,120.0,60,60,101,48,53,0,0,0,19,12,7
42476,NaN,17877_Viterbo,NaN,2,133.0,72,61,117,65,52,0,0,0,16,7,9
42477,NaN,17877_Viterbo,NaN,3,120.0,65,55,104,59,45,0,0,0,16,6,10
42478,NaN,17877_Viterbo,NaN,4,128.0,70,58,106,61,45,0,0,0,22,9,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42592,NaN,17877_Viterbo,NaN,Total grupo edad,8.0,4,4,8,4,4,0,0,0,0,0,0
42593,NaN,17877_Viterbo,100 a 104,101,2.0,1,1,2,1,1,0,0,0,0,0,0
42594,NaN,17877_Viterbo,NaN,102,1.0,0,1,1,0,1,0,0,0,0,0,0
42595,NaN,17877_Viterbo,NaN,103,1.0,0,1,1,0,1,0,0,0,0,0,0


Debido a la agrupación de columnas por área, se separan los datos por subconjuntos dependiendo del área y al final serán reunidos en un mismo dataframe con el área presente en las filas.

In [19]:
ls_df = []
area_type = ['total', 'cabecera', 'centro_poblado', 'rural_disperso']
for i, area in zip(range(4, 16, 3), area_type):
    df_3pm_with_area = pd.concat([df_3pm[['Unnamed: 2', 'Unnamed: 3']], df_3pm.iloc[:, i:i+3].assign(Area=area)], axis=1)
    ls_df.append(df_3pm_with_area)

for df in ls_df:
    df.rename(columns=dict(zip(list(df.columns.values[2:-1]), actual_headers[:])), inplace=True)

df_3pm = pd.concat(ls_df)
df_3pm.rename(columns={'Unnamed: 2': 'Grupo de edad', 'Unnamed: 3': 'Edad'}, inplace=True)
df_3pm

,Grupo de edad,Edad,Total,Hombre,Mujer,Area
42474,0 a 4,0,119.0,63,56,total
42475,NaN,1,120.0,60,60,total
42476,NaN,2,133.0,72,61,total
42477,NaN,3,120.0,65,55,total
42478,NaN,4,128.0,70,58,total
...,...,...,...,...,...,...
42592,NaN,Total grupo edad,0,0,0,rural_disperso
42593,100 a 104,101,0,0,0,rural_disperso
42594,NaN,102,0,0,0,rural_disperso
42595,NaN,103,0,0,0,rural_disperso


Se propaga el valor de 'Grupo de edad' Para completar los datos faltantes.

In [22]:
df_3pm['Grupo de edad'] = df_3pm['Grupo de edad'].ffill()
df_3pm

,Grupo de edad,Edad,Total,Hombre,Mujer,Area
42474,0 a 4,0,119.0,63,56,total
42475,0 a 4,1,120.0,60,60,total
42476,0 a 4,2,133.0,72,61,total
42477,0 a 4,3,120.0,65,55,total
42478,0 a 4,4,128.0,70,58,total
...,...,...,...,...,...,...
42592,95 a 99,Total grupo edad,0,0,0,rural_disperso
42593,100 a 104,101,0,0,0,rural_disperso
42594,100 a 104,102,0,0,0,rural_disperso
42595,100 a 104,103,0,0,0,rural_disperso


Se creará una copia del dataframe y se removerán las filas y columnas con resumenes de datos para el csv que no contiene resúmenes.

In [23]:
df_3pm_cleaned = df_3pm.copy()
df_3pm_cleaned.drop(['Total'], axis=1, inplace=True)
df_3pm_cleaned

,Grupo de edad,Edad,Hombre,Mujer,Area
42474,0 a 4,0,63,56,total
42475,0 a 4,1,60,60,total
42476,0 a 4,2,72,61,total
42477,0 a 4,3,65,55,total
42478,0 a 4,4,70,58,total
...,...,...,...,...,...
42592,95 a 99,Total grupo edad,0,0,rural_disperso
42593,100 a 104,101,0,0,rural_disperso
42594,100 a 104,102,0,0,rural_disperso
42595,100 a 104,103,0,0,rural_disperso


In [24]:
df_3pm_cleaned = df_3pm_cleaned.loc[(df_3pm_cleaned['Edad'] != 'Total grupo edad') & (df_3pm_cleaned['Area'] != 'total')]
df_3pm_cleaned

,Grupo de edad,Edad,Hombre,Mujer,Area
42474,0 a 4,0,53,42,cabecera
42475,0 a 4,1,48,53,cabecera
42476,0 a 4,2,65,52,cabecera
42477,0 a 4,3,59,45,cabecera
42478,0 a 4,4,61,45,cabecera
...,...,...,...,...,...
42590,95 a 99,97,0,0,rural_disperso
42591,95 a 99,98,0,0,rural_disperso
42593,100 a 104,101,0,0,rural_disperso
42594,100 a 104,102,0,0,rural_disperso


Los dataframes están listos para ser exportados a CSV.

In [25]:
df_3pm.to_csv(csv_total_path, index=False)
df_3pm_cleaned.to_csv(csv_path, index=False)